# LigandMPNN Inverse Folding

This notebook demonstrates how to use `run_ligandmpnn_sample` to design protein sequences conditioned on both backbone structure and nearby ligand atoms, and `run_ligandmpnn_score` to score sequence-structure compatibility in the same context. LigandMPNN extends ProteinMPNN by incorporating small molecule geometry into sequence design, enabling the creation of binding pocket sequences that are compatible with a specific ligand conformation. When no ligand atoms are present in the input structure, LigandMPNN behaves similarly to ProteinMPNN.


In [ ]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("ligandmpnn")
display_overview("ligandmpnn")
display_docs_section("ligandmpnn", "Background")

## Available tools

In [ ]:
display_available_tools("ligandmpnn")

### `run_ligandmpnn_sample`

LigandMPNN analyzes the backbone coordinates and any ligand atoms present in the structure to generate sequences optimized for the target fold and binding environment. We use GFP as a minimal runnable example. For ligand-aware design, provide a structure file that includes HETATM records for the bound ligand of interest. You can optionally fix residue positions to preserve critical interactions and exclude specific amino acids from the designed sequences.

In [ ]:
from proto_tools import (
    InverseFoldingStructureInput,
    LigandMPNNSampleConfig,
    LigandMPNNSampleInput,
    LigandMPNNScoringConfig,
    LigandMPNNScoringInput,
    SequenceStructurePair,
    run_ligandmpnn_sample,
    run_ligandmpnn_score,
)
from proto_tools.entities.structures.examples import get_gfp_structure


In [ ]:
# Display input docs
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_sample")

# Load GFP structure and configure design constraints
gfp_structure = get_gfp_structure()
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    fixed_positions={"A": [2, 3, 4]},
)
inputs = LigandMPNNSampleInput(inputs=[struct_input])

In [ ]:
# Display config docs
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_sample")

# Configure sampling | see docs above for all fields
config = LigandMPNNSampleConfig(
    num_sequences_per_structure=2,  # Generate 2 sequence designs
    temperature=0.1,                # Conservative designs
    excluded_amino_acids=["C"],     # Exclude cysteine
    seed=42,
    device="cuda",                  # Change to "cpu" if no GPU available
)

In [ ]:
# Run the sampling function
result = run_ligandmpnn_sample(inputs, config)

In [ ]:
# Display output docs
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_sample")

# Print designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"Structure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        preview = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {preview}")
        print(f"            Length: {len(seq)} residues")


### `run_ligandmpnn_score`

Use `run_ligandmpnn_score` to compute LigandMPNN log-likelihood and perplexity for a sequence in a structure context. This example scores the first design produced above against the GFP backbone.


In [ ]:
display_api_reference("ligandmpnn", "input", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "config", "run_ligandmpnn_score")
display_api_reference("ligandmpnn", "output", "run_ligandmpnn_score")

score_inputs = LigandMPNNScoringInput(
    sequence_structure_pairs=[
        SequenceStructurePair(
            sequence=result.designed_sequences[0].sequences[0],
            structure=gfp_structure,
        )
    ]
)
score_result = run_ligandmpnn_score(
    score_inputs,
    LigandMPNNScoringConfig(seed=42, return_logits=False),
)

score = score_result.scores[0]
print(f"Perplexity: {score.perplexity:.3f}")
print(f"Average log-likelihood: {score.avg_log_likelihood:.3f}")


## Export Results

Designed sequences can be saved to standard file formats for downstream analysis. JSON format is convenient for preserving the full LigandMPNN metrics alongside the sequences, while FASTA format is useful for feeding results into other sequence analysis tools.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="ligandmpnn_designs", export_path=output_dir, file_format="json")
score_result.export(name="ligandmpnn_scores", export_path=output_dir, file_format="json")
print(f"Results exported to {output_dir}")